# TRACER — Module 2: Vision-Language Model Integration

Demonstrates the `src/vlm` package built in Module 2: loading CLIP through the `CLIPVLM`
class, encoding images and text, computing similarity, extracting vision-encoder attentions,
and a first real zero-shot inference example.

**Run this after Module 1's bootstrap notebook** — it assumes the repo is already cloned to
`/content/tracers` with dependencies installed.

In [ ]:
%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")  # so `from src.vlm import ...` resolves

from src.vlm import CLIPVLM, VLMConfig
import torch

print("Imports OK.")


## Load the model through the CLIPVLM wrapper

In [ ]:
config = VLMConfig()  # device auto-detected (cuda if available, else cpu)
vlm = CLIPVLM(config)

print(f"Loaded on device: {vlm.device}")
print(f"Vision encoder layers: {vlm.model.config.vision_config.num_hidden_layers}")


## Encode an image

In [ ]:
from PIL import Image
import numpy as np

# A simple synthetic image for a first smoke test — replace with a real photo below.
dummy_array = (np.random.rand(224, 224, 3) * 255).astype("uint8")
dummy_image = Image.fromarray(dummy_array)

pixel_values = vlm.preprocess_image(dummy_image)
image_embeds = vlm.encode_image(pixel_values)

print(f"pixel_values shape: {pixel_values.shape}")
print(f"image_embeds shape: {image_embeds.shape}")
print(f"embedding is L2-normalized: {torch.allclose(image_embeds.norm(dim=-1), torch.tensor([1.0]).to(vlm.device), atol=1e-4)}")


## Encode text

In [ ]:
text_embeds = vlm.encode_text(["a photo of a dog", "a photo of a cat", "a photo of a car"])
print(f"text_embeds shape: {text_embeds.shape}")


## First real inference: zero-shot classification

This is the classic CLIP demo — given one image and several candidate captions, the
similarity scores tell you which caption the model thinks best matches the image. Upload
your own image to make this meaningful (use the folder icon on the left sidebar, or the
cell below to upload directly).

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick any image file from your computer
image_path = list(uploaded.keys())[0]
real_image = Image.open(image_path).convert("RGB")

candidate_labels = ["a dog", "a cat", "a car", "a person", "a building", "food"]

pixel_values = vlm.preprocess_image(real_image)
image_embeds = vlm.encode_image(pixel_values)
text_embeds = vlm.encode_text(candidate_labels)

similarities = vlm.similarity(image_embeds, text_embeds).squeeze()
probs = torch.softmax(similarities * 100, dim=0)  # CLIP convention: scale before softmax

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(real_image); ax[0].axis("off"); ax[0].set_title("Uploaded image")
ax[1].barh(candidate_labels, probs.detach().cpu().numpy())
ax[1].set_xlabel("Probability")
ax[1].set_title("Zero-shot classification")
plt.tight_layout()
plt.show()

best_idx = probs.argmax().item()
print(f"Best match: '{candidate_labels[best_idx]}' ({probs[best_idx]:.1%} confidence)")


## Extracting vision-encoder attentions

This is the piece Module 5 (detection) and Module 6 (attribution) will build on — confirming
it works correctly now, in isolation, before anything depends on it.

In [ ]:
attentions = vlm.get_vision_attentions(pixel_values)

print(f"Number of layers: {len(attentions)}")
print(f"Shape of one layer's attention tensor: {attentions[0].shape}")
print("(batch, attention_heads, tokens, tokens)")


## Module 2 completion check

If every cell above ran without errors and the zero-shot classification picked a sensible
label for your uploaded image, Module 2 is functionally complete. Run the test suite to
confirm formally:

```bash
cd ai-engine
pytest -m integration -v
```

All tests should pass. Then continue to **Module 3 — Dataset Management**.